# EXP002 untouched Qwen3.5-2B baseline — merge and validate


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_REV = '9b7eedd'
REPO_ROOT = Path('/kaggle/working/spider')
subprocess.run(['git', 'clone', 'https://github.com/yogesh-dhande/spider.git', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))


In [ ]:
%pip install -q --progress-bar off -r requirements/experiment2-kaggle.txt


In [ ]:
from spider.workflow import find_prepared_data
prepared = find_prepared_data(
    '/kaggle/input'
)
os.environ['SPIDER_DATA_DIR'] = str(prepared)
print({'prepared_data': str(prepared)})


In [ ]:
from spider.workflow import restore_evaluation_shards
labels = ['baseline-shard-00-of-08', 'baseline-shard-01-of-08', 'baseline-shard-02-of-08', 'baseline-shard-03-of-08', 'baseline-shard-04-of-08', 'baseline-shard-05-of-08', 'baseline-shard-06-of-08', 'baseline-shard-07-of-08']
restored = restore_evaluation_shards(
    ['/kaggle/input/notebooks/yogeshkd'], labels, REPO_ROOT
)
print({'restored_shards': [str(path) for path in restored]})


In [ ]:
from spider.merge import merge_evaluation_shards
predictions_path, metrics = merge_evaluation_shards(
    'configs/experiment2.yaml', 'baseline', labels,
    ['molmoweb', 'screenspot'], 'test'
)
completed = sum(1 for _ in predictions_path.open(encoding='utf-8'))
assert completed == 5272, completed
print({'event': 'baseline_merge_complete', 'completed': completed})
metrics
